In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: sim_02_sap (parameterized)
# Schedule: Every 15 min (incremental) OR Nightly 1:30 AM (snapshot)

import uuid
import random
import datetime as dt
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col, input_file_name

spark = SparkSession.builder.getOrCreate()

ROOT = LANDING
SEED = f"{ROOT}/_seed"

dbutils.widgets.dropdown("run_mode", "incremental", ["incremental", "snapshot"], "Run Mode")
RUN_MODE = dbutils.widgets.get("run_mode")

# ============================================================
# 1. SAFETY CHECK (Seed must exist for FK)
# ============================================================
try:
    machines = [r.asDict() for r in spark.read.json(f"{SEED}/machines").collect()]
    vendors = [r.asDict() for r in spark.read.json(f"{SEED}/vendors").collect()]
    materials = [r.asDict() for r in spark.read.json(f"{SEED}/materials").collect()]
    ccs = [r.asDict() for r in spark.read.json(f"{SEED}/cost_centers").collect()]
    if not machines: raise Exception("Seed empty")
except Exception as e:
    dbutils.notebook.exit(f"❌ SEED MISSING — run sim_00_seed first. Error: {e}")

machine_ids = [m["machine_id"] for m in machines]
vendor_ids = [v["vendor_id"] for v in vendors]
material_ids = [m["material_id"] for m in materials]
cc_ids = [c["cost_center_id"] for c in ccs]

run_ts = dt.datetime.now(dt.timezone.utc)
run_id = f"{run_ts:%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"

# ============================================================
# MODE 1: INCREMENTAL (Every 15 mins)
# ============================================================
if RUN_MODE == "incremental":
    print("📥 SAP INCREMENTAL mode starting...")

    # 1. ASSET MASTER (FK: machine_id)
    assets = [{
        "asset_id": f"AST-{random.randint(1000,9999)}",
        "machine_id": random.choice(machine_ids),
        "asset_class": random.choice(["HEAVY", "LIGHT", "TOOL"]),
        "book_value": float(f"{random.uniform(50000, 500000):.2f}"),
        "last_modified_ts": run_ts.isoformat()
    } for _ in range(random.randint(3, 6))]
    
    spark.createDataFrame(assets).write.mode("append").parquet(f"{ROOT}/sap/incremental/asset_master/{run_id}.parquet")
    print(f"  ✓ asset_master: {len(assets)} rows")

    # 2. VENDOR (No FK)
    vens = [{
        "vendor_id": f"VEN-{random.randint(100,999)}",
        "vendor_name": f"NewVendor_{random.randint(1,99)}",
        "country_code": random.choice(["IN", "US", "DE"]),
        "last_modified_ts": run_ts.isoformat()
    } for _ in range(random.randint(0, 2))]
    if vens:
        spark.createDataFrame(vens).write.mode("append").parquet(f"{ROOT}/sap/incremental/vendor/{run_id}.parquet")
        print(f"  ✓ vendor: {len(vens)} rows")

    # 3. PURCHASE ORDER (FK: vendor_id)
    pos = [{
        "po_number": f"PO-{random.randint(100000,999999)}",
        "vendor_id": random.choice(vendor_ids),
        "po_date": run_ts.date().isoformat(),
        "total_amount": float(f"{random.uniform(1000, 100000):.2f}"),
        "currency": "USD",
        "last_modified_ts": run_ts.isoformat()
    } for _ in range(random.randint(2, 4))]
    spark.createDataFrame(pos).write.mode("append").parquet(f"{ROOT}/sap/incremental/purchase_order/{run_id}.parquet")
    print(f"  ✓ purchase_order: {len(pos)} rows")

    # 4. PO LINE ITEM (FK: po_number, material_id)
    lines = []
    for po in pos:
        for ln in range(1, random.randint(1, 3) + 1):
            lines.append({
                "po_number": po["po_number"],
                "line_no": ln,
                "material_id": random.choice(material_ids),
                "qty": random.randint(1, 100),
                "unit_price": float(f"{random.uniform(10, 2000):.2f}")
            })
    if lines:
        spark.createDataFrame(lines).write.mode("append").parquet(f"{ROOT}/sap/incremental/po_line_item/{run_id}.parquet")
        print(f"  ✓ po_line_item: {len(lines)} rows")

    # 5. FINANCIAL POSTING (FK: asset_id)
    posts = [{
        "posting_id": f"POST-{random.randint(1000000,9999999)}",
        "asset_id": random.choice(assets)["asset_id"],
        "gl_account": f"GL-{random.randint(1000,9999)}",
        "amount": float(f"{random.uniform(100, 50000):.2f}"),
        "posting_date": run_ts.date().isoformat(),
        "last_modified_ts": run_ts.isoformat()
    } for _ in range(random.randint(2, 5))]
    spark.createDataFrame(posts).write.mode("append").parquet(f"{ROOT}/sap/incremental/financial_posting/{run_id}.parquet")
    print(f"  ✓ financial_posting: {len(posts)} rows")

    # 6. ASSET COST CENTER MAP (FK: asset_id, cost_center_id)
    acm = []
    for a in assets:
        for cc in random.sample(cc_ids, k=random.randint(1, 2)):
            acm.append({
                "asset_id": a["asset_id"],
                "cost_center_id": cc,
                "valid_from": "2023-01-01",
                "valid_to": "9999-12-31"
            })
    if acm:
        spark.createDataFrame(acm).write.mode("append").parquet(f"{ROOT}/sap/incremental/asset_cost_center_map/{run_id}.parquet")
        print(f"  ✓ asset_cost_center_map: {len(acm)} rows")


# ============================================================
# MODE 2: SNAPSHOT (Nightly for Delete Detection)
# ============================================================
else:
    print("📸 SAP SNAPSHOT mode starting (with deliberate row deletions)...")
    snapshot_date = run_ts.date().isoformat()

    def get_base_data(table_name, count):
        if table_name == "asset_master":
            return [{"asset_id": f"AST-{i:04d}", "machine_id": mid, "asset_class": "HEAVY", "book_value": 100000.0} 
                    for i, mid in enumerate(machine_ids[:count], start=1)]
        elif table_name == "vendor":
            return [{"vendor_id": v["vendor_id"], "vendor_name": v["vendor_name"], "country_code": "IN"} for v in vendors[:count]]
        elif table_name == "material_master":
            return [{"material_id": m["material_id"], "material_desc": m["material_desc"]} for m in materials[:count]]
        elif table_name == "purchase_order":
            return [{"po_number": f"PO-{i:05d}", "vendor_id": random.choice(vendor_ids), "total_amount": 5000.0} for i in range(1, count+1)]

    for table in ["asset_master", "vendor", "material_master", "purchase_order"]:
        data = get_base_data(table, 40)
        
        if len(data) > 2:
            n_drop = random.randint(1, 2)
            data = data[:-n_drop]
            print(f"  🗑️ {table}: dropped {n_drop} rows (simulating SAP delete)")

        spark.createDataFrame(data).coalesce(1).write.mode("overwrite") \
            .option("header", "true") \
            .csv(f"{ROOT}/sap/snapshot/{table}/snapshot_{snapshot_date}.csv")
        print(f"  ✓ {table}: {len(data)} rows")

print(f"\n✅ SAP {RUN_MODE} complete: {run_id}")